# Behaviour Dataset Processing

## Overview

This notebook processes a single National Travel Survey (NTS) behavioural dataset.

The objective is to clean, harmonise, and transform mobility behaviour information into a standardised behavioural representation for the proposed mobility representation framework.

The processed behavioural data will be exported as a standardised JSON file for subsequent integration with the demographic layer.

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |NTS0611: Average number of trips and distance travelled by sex, age and purpose: England, 2002 onwards |
| Source |National Travel Survey|
| Geography | England |
| Input File |nts0611.ods|
| Output File |nts0611a_trip_purpose_probability.json|
| Notebook Author | Peter |
| Last Updated |2026-08-05|

## 1. Import

In [12]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [13]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 2. Configuration

In [14]:
from pathlib import Path

# File paths

input_file = "nts0611.ods"
output_file = "nts0611a_trip_purpose_probability.json"

raw_data_path = Path("../../data/nts/raw")
output_data_path = Path("../../data/nts/harmonised")

## 3. Load Dataset

In [15]:
df = pd.read_excel(
    raw_data_path / input_file,
    sheet_name="NTS0611a_trips",
    skiprows=5
)

df_processed = df.copy()

df_processed.head()

,Year,Sex [note 2],Age,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1],Other [note 1],All purposes,Unweighted sample size: individuals,Unweighted sample size: trips (thousands)
0,2002,All people,0 to 16,10.965709,0.601430,252.435245,41.212326,84.532346,158.556695,72.385670,153.334289,26.625333,51.058601,32.746723,11.683989,22.478070,23.582358,0.075075,942.273859,3084,52.582
1,2002,All people,17 to 20,220.875052,9.504783,121.107835,6.799904,158.791795,37.123523,70.541268,176.830659,92.210300,46.763126,16.793644,6.681502,10.534957,20.952200,7.288901,1002.799449,547,10.115
2,2002,All people,21 to 29,268.906081,36.833463,33.816748,37.922372,199.521342,76.936998,91.762702,138.665827,71.354870,47.723286,17.696085,9.364377,14.812031,15.517718,0.389899,1061.223796,1262,24.934
3,2002,All people,30 to 39,254.569086,69.237394,6.745660,115.046571,235.796917,131.253956,120.152378,120.541820,52.938785,43.707804,15.600966,9.101992,26.115206,31.236774,0.165421,1232.210729,2083,47.089
4,2002,All people,40 to 49,289.872426,70.669378,2.979659,80.692897,256.576587,139.028234,133.807504,105.243938,54.731065,45.300523,15.269404,12.874706,22.186667,49.604000,0.000000,1278.836987,2011,47.071


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [16]:
df_processed.info()

print(f"Shape: {df_processed.shape}")

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 621 entries, 0 to 620
Data columns (total 21 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Year                                       621 non-null    int64  
 1   Sex [note 2]                               621 non-null    object 
 2   Age                                        621 non-null    object 
 3   Commuting                                  621 non-null    float64
 4   Business                                   621 non-null    float64
 5   Education                                  621 non-null    float64
 6   Escort education                           621 non-null    float64
 7   Shopping                                   621 non-null    float64
 8   Other escort                               621 non-null    float64
 9   Personal business                          621 non-null    float64
 10  Visiting friends at privat

Index(['Year', 'Sex [note 2]', 'Age', 'Commuting', 'Business', 'Education',
       'Escort education', 'Shopping', 'Other escort', 'Personal business',
       'Visiting friends at private home', 'Visiting friends elsewhere',
       'Entertainment or public activity', 'Sport: participate',
       'Holiday: base', 'Day trip', 'Just walk [note 1]', 'Other [note 1]',
       'All purposes', 'Unweighted sample size: individuals',
       'Unweighted sample size: trips (thousands)'],
      dtype='object')

## 5. Data Cleaning

Check the dataset quality before feature processing.

In [17]:
# Missing values
df_processed.isna().sum()

Year                                         0
Sex [note 2]                                 0
Age                                          0
Commuting                                    0
Business                                     0
Education                                    0
Escort education                             0
Shopping                                     0
Other escort                                 0
Personal business                            0
Visiting friends at private home             0
Visiting friends elsewhere                   0
Entertainment or public activity             0
Sport: participate                           0
Holiday: base                                0
Day trip                                     0
Just walk [note 1]                           0
Other [note 1]                               0
All purposes                                 0
Unweighted sample size: individuals          0
Unweighted sample size: trips (thousands)    0
dtype: int64

In [18]:
# Duplicate records
df_processed.duplicated().sum()

np.int64(0)

In [19]:
# Summary statistics
df_processed.describe(include="all")

,Year,Sex [note 2],Age,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1],Other [note 1],All purposes,Unweighted sample size: individuals,Unweighted sample size: trips (thousands)
count,621.000000,621,621,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000,621.000000
unique,NaN,3,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,All people,0 to 16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,207,69,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2013.000000,NaN,NaN,150.379616,28.733109,55.658725,46.449061,194.154424,77.792722,93.579203,97.146814,48.484949,49.318892,14.930747,11.008467,29.568155,56.714469,0.817893,954.737244,2209.887279,38.880171
std,6.638597,NaN,NaN,97.886046,22.985855,85.655831,47.611583,79.688824,37.624159,32.206517,27.633038,13.165428,10.805266,8.061770,3.130138,10.327317,30.035622,1.813242,157.844601,3126.919184,55.806425
min,2002.000000,NaN,NaN,0.231273,0.000000,0.000000,0.632550,16.576109,11.487536,15.326190,41.755641,11.687297,10.063343,0.935107,1.475877,7.212624,7.485040,0.000000,481.988774,83.000000,0.912000
25%,2007.000000,NaN,NaN,74.655027,8.688014,0.688460,11.909900,146.998879,47.087801,68.945941,77.088552,42.219073,44.039483,9.455565,9.016835,23.194998,36.674701,0.000000,853.343232,843.000000,13.607000
50%,2013.000000,NaN,NaN,161.633569,25.911854,4.040008,32.504803,195.314402,76.152513,91.350252,92.701544,49.512697,49.755368,13.290438,11.090212,29.161981,49.414133,0.155644,966.865495,1123.000000,21.093000
75%,2019.000000,NaN,NaN,232.396284,43.909061,66.725865,56.539141,250.306737,103.656512,117.586466,113.132331,55.767662,55.888111,18.323619,12.631025,35.285941,67.988780,0.498796,1056.769774,1991.000000,35.585000


## 6. Feature Processing

### Step 6.1

Process the behavioural information required for the mobility representation framework.

The processing workflow in this section depends on the specific NTS dataset.

In [20]:
# The study year is 2024 only, so we can add a column for the year and set it to 2024 for all rows.
df_processed = df_processed[df_processed["Year"] == 2024].copy()

nts0611a_selected_columns = [
    "Year",
    "Sex [note 2]",
    "Age",
    "Commuting",
    "Business",
    "Education",
    "Escort education",
    "Shopping",
    "Other escort",
    "Personal business",
    "Visiting friends at private home",
    "Visiting friends elsewhere",
    "Entertainment or public activity",
    "Sport: participate",
    "Holiday: base",
    "Day trip",
    "Just walk [note 1]"]

df_processed = df_processed[nts0611a_selected_columns]

df_processed.rename(
    columns={
        "Sex [note 2]": "Sex",
        "Escort education": "Escort Education",
        "Other escort": "Other Escort",
        "Personal business": "Personal Business",
        "Visiting friends at private home": "Visiting Friends at Private Home",
        "Visiting friends elsewhere": "Visiting Friends Elsewhere",
        "Entertainment or public activity": "Entertainment or Public Activity",
        "Sport: participate": "Sport: Participate",
        "Holiday: base": "Holiday: Base",
        "Day trip": "Day Trip",
        "Just walk [note 1]": "Just Walk"
    },
    inplace=True
)


In [21]:
df_processed["Sex"] = (
    df_processed["Sex"]
    .replace({
        "Females": "Female",
        "Males": "Male",
    })
)

In [22]:
from ontology.FRAMEWORK_TRIP_PURPOSE import FRAMEWORK_TRIP_PURPOSE

In [23]:
trip_purpose_columns = FRAMEWORK_TRIP_PURPOSE

df_processed["Total Trips"] = df_processed[trip_purpose_columns].sum(axis=1)

In [24]:
df_processed[["Age", "Sex", "Total Trips"]].head()

,Age,Sex,Total Trips
594,0 to 16,All people,804.003353
595,17 to 20,All people,728.974963
596,21 to 29,All people,849.416147
597,30 to 39,All people,1022.621548
598,40 to 49,All people,1047.105936


In [25]:
trip_purpose_probability = df_processed.copy()

trip_purpose_probability[trip_purpose_columns] = (trip_purpose_probability[trip_purpose_columns].div(trip_purpose_probability["Total Trips"], axis=0))

In [26]:
if np.all(np.isclose(
    trip_purpose_probability[trip_purpose_columns].sum(axis=1),
    1.0
)):
    print("All trip purpose probabilities sum to 1.")
else:
    print("Some rows do not sum to 1.")

All trip purpose probabilities sum to 1.


## Step 6.2 - Age Harmonisation

In [27]:
age_dataset = pd.read_excel(
    "../../data/demographic/raw/age/TS007_Age.xlsx"
)

In [28]:
processed_age_dataset = age_dataset[["Lower tier local authorities", "Age (86 categories) Code", "Observation"]].copy()

In [29]:
study_area = "Manchester"

age_single = processed_age_dataset[processed_age_dataset["Lower tier local authorities"] == study_area].copy()
age_single.head()

,Lower tier local authorities,Age (86 categories) Code,Observation
20812,Manchester,0,6581
20813,Manchester,1,7057
20814,Manchester,2,6819
20815,Manchester,3,6782
20816,Manchester,4,7139


In [30]:
age_mapping = {
    "0-4": [{"nts": "0 to 16", "start": 0, "end": 4}],

    "5-11": [{"nts": "0 to 16", "start": 5, "end": 11}],

    "12-17": [
        {"nts": "0 to 16", "start": 12, "end": 16},
        {"nts": "17 to 20", "start": 17, "end": 17},
    ],

    "18-22": [
        {"nts": "17 to 20", "start": 18, "end": 20},
        {"nts": "21 to 29", "start": 21, "end": 22},
    ],

    "23-34": [
        {"nts": "21 to 29", "start": 23, "end": 29},
        {"nts": "30 to 39", "start": 30, "end": 34},
    ],

    "35-49": [
        {"nts": "30 to 39", "start": 35, "end": 39},
        {"nts": "40 to 49", "start": 40, "end": 49},
    ],

    "50-64": [
        {"nts": "50 to 59", "start": 50, "end": 59},
        {"nts": "60 to 69", "start": 60, "end": 64},
    ],

    "65-74": [
        {"nts": "60 to 69", "start": 65, "end": 69},
        {"nts": "70 and over", "start": 70, "end": 74},
    ],
    
    "75+": [{"nts": "70 and over", "start": 75, "end": None}],
}

### Step  6.3: Create a generator to calculate population

In [31]:
def get_population_count(start_age, end_age=None):
    #Open-ended age groups (e.g. 75+) are handled by aggregating all population counts above the specified lower bound.

    if end_age is None:

        return age_single[
            age_single["Age (86 categories) Code"] >= start_age
        ]["Observation"].sum()

    return age_single[
        age_single["Age (86 categories) Code"].between(start_age, end_age)
    ]["Observation"].sum()

In [32]:
get_population_count(0,4)

np.int64(34378)

### Step  6.4: Create a generator to calculate population weights

In [33]:
def get_weights_for_framework_age(framework_age):
    
    #Calculate population-based interpolation weights for one framework age group.

    age_overlap = age_mapping[framework_age]

    populations = []

    for overlap in age_overlap:
        population = get_population_count(
            overlap["start"],
            overlap["end"]
        )
        populations.append(population)

    total_population = sum(populations)

    overlap_weights = [
        population / total_population
        for population in populations
    ]

    return overlap_weights

In [34]:
get_weights_for_framework_age("12-17")

[np.float64(0.8465046329985931), np.float64(0.15349536700140687)]

### Step 6.5 Combine the two generators

In [35]:
def harmonise_one_FRAMEWORK_AGE_GROUP(trip_purpose_probability, framework_age, Sex, trip_purpose_columns):
    
    """Harmonise one framework age group using population-weighted interpolation."""

    age_overlap = age_mapping[framework_age]

    overlap_weights = get_weights_for_framework_age(
        framework_age
    )

    harmonised_trip_purpose_probability = pd.Series(
        0.0,
        index=trip_purpose_columns
    )

    for overlap, weight in zip(age_overlap, overlap_weights):

        nts_row = trip_purpose_probability[
            (trip_purpose_probability["Sex"] == Sex)
            &
            (trip_purpose_probability["Age"] == overlap["nts"])
        ].iloc[0]

        harmonised_trip_purpose_probability += (
            weight * nts_row[trip_purpose_columns]
        )

    harmonised_row = {
        "Year": 2024,
        "Sex": Sex,
        "Age": framework_age
    }

    harmonised_row.update(
        harmonised_trip_purpose_probability.to_dict()
    )

    harmonised_row["prob_sum"] = (
        harmonised_trip_purpose_probability.sum()
    )

    return harmonised_row

In [36]:
FRAMEWORK_AGE_GROUPs = [
    "0-4",
    "5-11",
    "12-17",
    "18-22",
    "23-34",
    "35-49",
    "50-64",
    "65-74",
    "75+"
]

Sex_groups = [
    "Female",
    "Male"
]

harmonised_rows = []

for Sex in Sex_groups:

    for framework_age in FRAMEWORK_AGE_GROUPs:

        harmonised_rows.append(

            harmonise_one_FRAMEWORK_AGE_GROUP(
                trip_purpose_probability,
                framework_age,
                Sex,
                trip_purpose_columns
            )

        )

harmonised_trip_purpose_probability = pd.DataFrame(
    harmonised_rows
)

harmonised_trip_purpose_probability = (
    harmonised_trip_purpose_probability[
        ["Year", "Sex", "Age"]
        + trip_purpose_columns
    ]
)


display(harmonised_trip_purpose_probability)

,Year,Sex,Age,Commuting,Business,Education,Escort Education,Shopping,Other Escort,Personal Business,Visiting Friends at Private Home,Visiting Friends Elsewhere,Entertainment or Public Activity,Sport: Participate,Holiday: Base,Day Trip,Just Walk
0,2024,Female,0-4,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662
1,2024,Female,5-11,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662
2,2024,Female,12-17,0.026672,0.001690,0.304515,0.038551,0.056038,0.125005,0.074419,0.108590,0.051023,0.074196,0.031747,0.015177,0.040354,0.052022
3,2024,Female,18-22,0.180319,0.015662,0.117592,0.019347,0.189261,0.036292,0.071061,0.109007,0.080139,0.057312,0.007461,0.017746,0.035592,0.063210
4,2024,Female,23-34,0.187671,0.030108,0.019513,0.088612,0.184447,0.069928,0.074421,0.086093,0.055916,0.059392,0.006704,0.017174,0.035667,0.084355
5,2024,Female,35-49,0.138959,0.030173,0.002271,0.170156,0.180556,0.100192,0.075522,0.064854,0.043113,0.053496,0.009181,0.011911,0.036452,0.083165
6,2024,Female,50-64,0.131147,0.032095,0.000690,0.039488,0.222626,0.066860,0.105210,0.088050,0.052823,0.062339,0.008032,0.013860,0.046023,0.130756
7,2024,Female,65-74,0.045730,0.013056,0.000397,0.021163,0.285039,0.047241,0.141740,0.109224,0.070948,0.075429,0.008485,0.013501,0.049347,0.118700
8,2024,Female,75+,0.011872,0.003727,0.000087,0.012801,0.324676,0.032908,0.159272,0.113154,0.082330,0.078310,0.007110,0.012737,0.047206,0.113809
9,2024,Male,0-4,0.003106,0.000525,0.333408,0.047855,0.023888,0.139715,0.068052,0.093075,0.037175,0.067786,0.065606,0.012175,0.048738,0.058896


## 7. Validation

Validate the processed dataset before exporting.

In [37]:
# Missing values
harmonised_trip_purpose_probability.isna().sum()

Year                                0
Sex                                 0
Age                                 0
Commuting                           0
Business                            0
Education                           0
Escort Education                    0
Shopping                            0
Other Escort                        0
Personal Business                   0
Visiting Friends at Private Home    0
Visiting Friends Elsewhere          0
Entertainment or Public Activity    0
Sport: Participate                  0
Holiday: Base                       0
Day Trip                            0
Just Walk                           0
dtype: int64

In [38]:
# Duplicate records
harmonised_trip_purpose_probability.duplicated().sum()

np.int64(0)

In [39]:
# AI-assisted implementation using ChatGPT
# See AI_PROMPTS.md
probability_columns = harmonised_trip_purpose_probability.columns[3:]

row_sum = harmonised_trip_purpose_probability[probability_columns].sum(axis=1)

display(row_sum)

assert np.allclose(row_sum, 1.0, atol=1e-8), (
    "Trip-purpose distributions do not sum to one."
)

print("PASS: All trip-purpose distributions sum to one.")

0     1.0
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
12    1.0
13    1.0
14    1.0
15    1.0
16    1.0
17    1.0
dtype: float64

PASS: All trip-purpose distributions sum to one.


## 8. Export

Export the processed dataset for later integration.

In [40]:
output_data_path.mkdir(parents=True, exist_ok=True)

harmonised_trip_purpose_probability.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## 9. Summary

### Input

- National Travel Survey (NTS)

### Output

- Standardised behavioural JSON

### Processing Completed

- Dataset cleaned
- Behaviour processed
- Validation passed
- JSON exported